In [37]:
import numpy as np 
import pandas as pd 

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer


In [38]:
df = pd.DataFrame({ 
    "Employee_ID": [101, 102, 103, 104, 105, 106, 107, 108], 
    "Department":  ["Sales", "IT", "HR", "IT", "Sales", "Finance", "HR", "Sales"], 
    "Job_Role":    ["Executive", "Developer", "Recruiter", "Analyst", "Manager", "Analyst", 
"HRBP", "Executive"], 
    "Gender":      ["M", "F", "F", "M", "M", "F", "F", "M"], 
    "Education":   ["Bachelors", "Masters", "Bachelors", "PhD", "Masters", "Bachelors", 
"Masters", "Bachelors"], 
    "Date_of_Joining": ["2020-06-15", "2019-02-10", "2021-09-01", "2018-12-05", "2022-01-20", 
"2017-07-30", "2020-11-18", "2016-04-12"], 
    "Monthly_Salary": [45000, 90000, 52000, 78000, 120000, 65000, 70000, 48000], 
    "Overtime_Flag":  ["Yes", "No", "No", "Yes", "Yes", "No", "No", "Yes"], 
    "Exit_Interview_Comments": [ 
        "Need better growth opportunities", 
        "Happy with team and work", 
        "Work-life balance issues", 
        "Good learning but high workload", 
        "Better offer, leaving", 
        "Satisfied overall", 
        "No clear career path", 
        "Commute too long, tired" 
    ], 
    "Attrition_Flag": ["Yes", "No", "Yes", "No", "Yes", "No", "Yes", "No"] 
})
print(df)

   Employee_ID Department   Job_Role Gender  Education Date_of_Joining  \
0          101      Sales  Executive      M  Bachelors      2020-06-15   
1          102         IT  Developer      F    Masters      2019-02-10   
2          103         HR  Recruiter      F  Bachelors      2021-09-01   
3          104         IT    Analyst      M        PhD      2018-12-05   
4          105      Sales    Manager      M    Masters      2022-01-20   
5          106    Finance    Analyst      F  Bachelors      2017-07-30   
6          107         HR       HRBP      F    Masters      2020-11-18   
7          108      Sales  Executive      M  Bachelors      2016-04-12   

   Monthly_Salary Overtime_Flag           Exit_Interview_Comments  \
0           45000           Yes  Need better growth opportunities   
1           90000            No          Happy with team and work   
2           52000            No          Work-life balance issues   
3           78000           Yes   Good learning but high 

In [39]:
df["Date_of_Joining"] = pd.to_datetime(df["Date_of_Joining"])
as_of = pd.Timestamp("2026-01-01") 
df["Tenure_Months"] = ((as_of - df["Date_of_Joining"]).dt.days / 30.44).round(1)
df['Tenure_Months']

0     66.6
1     82.7
2     52.0
3     84.9
4     47.4
5    101.1
6     61.4
7    116.7
Name: Tenure_Months, dtype: float64

In [40]:
df["Join_year"] = df["Date_of_Joining"].dt.year
df["Join_month"] = df["Date_of_Joining"].dt.month

In [41]:
le = LabelEncoder()
df["Attrition_label"] = le.fit_transform(df["Attrition_Flag"])
df["Gender_label"] = le.fit_transform(df["Gender"])
df["Overtime_label"] = le.fit_transform(df["Overtime_Flag"])

In [42]:
df.drop(["Gender"], axis=1, inplace= True)
df.drop(["Attrition_Flag"], axis=1, inplace= True)
df.drop(["Overtime_Flag"], axis=1, inplace= True)

In [54]:
ohe = OneHotEncoder(handle_unknown="ignore" , sparse_output= False)
ohe_features = ["Department", "Job_Role", "Gender_label", "Overtime_label"] 

preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe , ohe_features)
    ],
    remainder= "passthrough"
)
X = df[[ 
    "Department", "Job_Role", "Gender_label", "Overtime_label", 
    "Monthly_Salary", "Tenure_Months"]]

X_encoded = preprocessor.fit_transform(X)
ohe_feature_names = preprocessor.named_transformers_["ohe"].get_feature_names_out(ohe_features) 
numeric_feature_names = ["Monthly_Salary", "Tenure_Months"]
final_feature_names = list(ohe_feature_names) + numeric_feature_names
X_encoded_df = pd.DataFrame(X_encoded, columns= final_feature_names)

In [55]:
X_encoded_df

,Department_Finance,Department_HR,Department_IT,Department_Sales,Job_Role_Analyst,Job_Role_Developer,Job_Role_Executive,Job_Role_HRBP,Job_Role_Manager,Job_Role_Recruiter,Gender_label_0,Gender_label_1,Overtime_label_0,Overtime_label_1,Monthly_Salary,Tenure_Months
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,45000.0,66.6
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,90000.0,82.7
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,52000.0,52.0
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,78000.0,84.9
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,120000.0,47.4
5,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,65000.0,101.1
6,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,70000.0,61.4
7,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,48000.0,116.7


In [56]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{fea